# Somali News Dataset — Advanced Cleaning Notebook

This notebook is designed to be added after you already collected and combined your raw Somali news dataset.

It performs:

1. Raw dataset loading
2. Deep quality analysis
3. Site-specific cleaning
4. General text cleaning
5. Duplicate detection
6. Suspicious article review
7. Before/after inspection
8. Final cleaned dataset export

Expected input file:

```text
data/combined_raw_data.csv
```

The notebook will also try these fallback locations:

```text
combined_raw_data.csv
data/combined/combined_raw_data.csv
/mnt/data/combined_raw_data.csv
```

Final outputs:

```text
data/processed/cleaned_dataset.csv
data/processed/cleaned_dataset.xlsx
data/processed/suspicious_rows_for_review.csv
data/processed/suspicious_rows_for_review.xlsx
```

## Cell 1 — Setup and Load Combined Raw Dataset

In [1]:
# CELL 1 — SETUP AND LOAD COMBINED RAW DATASET
# ============================================

import pandas as pd
import numpy as np
import re
import html
import hashlib
from pathlib import Path
from difflib import SequenceMatcher

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 250)

# Try common dataset locations
POSSIBLE_INPUT_PATHS = [
    Path("data/combined_raw_data.csv"),
    Path("combined_raw_data.csv"),
    Path("data/combined/combined_raw_data.csv"),
    Path("/mnt/data/combined_raw_data.csv"),
]

INPUT_PATH = None

for path in POSSIBLE_INPUT_PATHS:
    if path.exists():
        INPUT_PATH = path
        break

if INPUT_PATH is None:
    raise FileNotFoundError(
        "❌ Could not find combined_raw_data.csv. "
        "Put it in data/combined_raw_data.csv or in the same folder as this notebook."
    )

raw_df = pd.read_csv(INPUT_PATH, dtype=str, keep_default_na=False)

# Standardize column names
raw_df.columns = (
    raw_df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Rename common scraper columns to expected names
rename_map = {
    "title": "headline",
    "content": "body",
    "article_title": "headline",
    "article_body": "body",
    "link": "url",
    "source_site": "site",
    "source": "site",
}

raw_df = raw_df.rename(columns={k: v for k, v in rename_map.items() if k in raw_df.columns})

# Ensure required columns exist
for col in ["site", "category", "url", "headline", "body"]:
    if col not in raw_df.columns:
        raw_df[col] = ""

# If site/category are missing, try to infer from source_file
if "source_file" in raw_df.columns:
    def infer_site_category_from_file(filename):
        stem = Path(str(filename)).stem.strip().lower()

        if "__" in stem:
            site, category = stem.split("__", 1)
        else:
            parts = stem.split("_")
            category = parts[-1] if len(parts) > 1 else ""
            site = "_".join(parts[:-1]) if len(parts) > 1 else stem

        return site.strip(), category.strip()

    missing_site = raw_df["site"].astype(str).str.strip().eq("")
    missing_category = raw_df["category"].astype(str).str.strip().eq("")

    inferred = raw_df["source_file"].apply(infer_site_category_from_file)
    raw_df.loc[missing_site, "site"] = inferred.apply(lambda x: x[0])
    raw_df.loc[missing_category, "category"] = inferred.apply(lambda x: x[1])

# Normalize important metadata
raw_df["site"] = raw_df["site"].astype(str).str.strip().str.lower()
raw_df["category"] = raw_df["category"].astype(str).str.strip().str.lower()
raw_df["url"] = raw_df["url"].astype(str).str.strip()
raw_df["headline"] = raw_df["headline"].astype(str).str.strip()
raw_df["body"] = raw_df["body"].astype(str).str.strip()

print("✅ Raw dataset loaded successfully.")
print(f"📁 Input path: {INPUT_PATH.resolve()}")
print(f"📊 Rows: {len(raw_df):,}")
print(f"📊 Columns: {len(raw_df.columns):,}")

display(raw_df.head())

✅ Raw dataset loaded successfully.
📁 Input path: E:\somali_cleaner\data\combined_raw_data.csv
📊 Rows: 35,797
📊 Columns: 9


,id,site,category,url,headline,body,scraped_at,word_count,source_file
0,bbc_somali__ciyaaro__000001,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/clyp28rnzw0o,Maxay dalalku ka faa'idaan marka ay u soo baxaan ka qeybgalka Koobka Adduunka?,"Xigashada Sawirka, Getty Images Dalalka si guul leh ugu soo baxa Koobka Adduunka ee FIFA waxay helaan sharaf weyn oo ay dunidu u hayso, sidoo kale faa'iidooyin dhaqaale iyo farxad weyn ayaa soo wajahda shacabkooda. Tani ma aha oo keliya guul ciya...",2026-05-19 20:30:29,656,bbc_somali__ciyaaro.csv
1,bbc_somali__ciyaaro__000002,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/cd9plkkj1pdo,Maxaa ka jira in ciyaartooyda kooxdii caanka ahayd ee Real Madrid mid mid isu dilayaan?,"Xigashada Sawirka, Getty Images Kooxda Real Madrid ayaa toddobaadkan gashay xaalad qalalaase weyn ah, xilli ay ahayd in dhammaan diiradda lagu saaro kulanka adag ee El Clasico ee ay la ciyaarayaan kooxda ay sida weyn u xafiiltamaan ee FC Barcelon...",2026-05-19 20:30:33,809,bbc_somali__ciyaaro.csv
2,bbc_somali__ciyaaro__000003,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/cjdplv01z52o,Ciyaaryahanada ugu sarreeya Afrika ee aan ka qayb galayn Koobka Adduunka 2026,"Xigashada Sawirka, Getty Images Iyada oo ay soo dhowaatay isku-aadka Koobka Adduunka, dalal badan, gaar ahaan kuwa ka qayb galaya koobka ayaa bilaabay qorshahooda inay dhowaan safraan. Sannadka, 48 dal ayaa ka qayb galaya tartanka oo ay si wadaji...",2026-05-19 20:30:36,616,bbc_somali__ciyaaro.csv
3,bbc_somali__ciyaaro__000004,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/c332veek778o,Maxaa ugu wacan Arsenal inaysan weligeed ku guulaysan Champions League,"Xigashada Sawirka, Getty Images Arsenal waligeed kuma guuleysan Champions League iyo Europa League. Hadda waxay soo gaartay Semifinalka, waxayna barbarro 1-1 ah la gashay Atletico Madrid oo ay marti ugu ahayd gurigeeda lugtii hore ee labada cayaa...",2026-05-19 20:30:39,848,bbc_somali__ciyaaro.csv
4,bbc_somali__ciyaaro__000005,bbc_somali,ciyaaro,https://www.bbc.com/somali/articles/cly07pjzdm0o,Dhamaan xogta iyo macluumaadka aad u baahan tahay in aad ka ogaato koobka kubadda cagta adduunka ee 2026,"Xigashada Sawirka, Getty Images Iyadoo dhamaan kooxaha ka qayb galaya koobka adduunka hadda la og yahay, indhaha ayaa u soo jeestay tartankii ugu weynaa ee Koobka Adduunka ee abid la qabto. Lixdii kooxood ee ugu dambeeyay ayaa dhawaan u soo baxay...",2026-05-19 20:30:42,1731,bbc_somali__ciyaaro.csv


## Cell 2 — Deep Raw Dataset Quality Report

In [2]:
# CELL 2 — DEEP RAW DATA QUALITY REPORT
# =====================================

df_report = raw_df.copy()

for col in ["site", "category", "url", "headline", "body"]:
    if col not in df_report.columns:
        df_report[col] = ""

df_report["site"] = df_report["site"].astype(str).str.strip().str.lower()
df_report["category"] = df_report["category"].astype(str).str.strip().str.lower()
df_report["body_words"] = df_report["body"].astype(str).str.split().str.len()
df_report["body_chars"] = df_report["body"].astype(str).str.len()
df_report["headline_chars"] = df_report["headline"].astype(str).str.len()
df_report["link_count"] = df_report["body"].astype(str).str.count(r"https?://|www\.")

print("═" * 80)
print("RAW DATASET OVERVIEW")
print("═" * 80)
print(f"Total rows: {len(df_report):,}")
print(f"Total sources: {df_report['site'].nunique():,}")
print(f"Total categories: {df_report['category'].nunique():,}")

print("\n📌 Rows by source:")
display(df_report["site"].value_counts().rename_axis("site").reset_index(name="rows"))

print("\n📌 Rows by category:")
display(df_report["category"].value_counts().rename_axis("category").reset_index(name="rows"))

print("\n📌 Missing / empty values:")
missing_report = []

for col in ["url", "headline", "body", "category", "site"]:
    missing_report.append({
        "column": col,
        "empty_count": int((df_report[col].astype(str).str.strip() == "").sum()),
        "empty_percent": round((df_report[col].astype(str).str.strip() == "").mean() * 100, 2),
    })

display(pd.DataFrame(missing_report))

print("\n📌 Body length statistics by source:")
display(
    df_report
    .groupby("site")["body_words"]
    .describe()
    .round(1)
    .reset_index()
)

print("\n📌 Duplicate report:")
duplicate_report = {
    "duplicate_url_rows": int(df_report[df_report["url"].duplicated(keep=False)].shape[0]),
    "unique_duplicate_urls": int(df_report[df_report["url"].duplicated(keep=False)]["url"].nunique()),
    "duplicate_exact_body_rows": int(df_report[df_report["body"].duplicated(keep=False)].shape[0]),
    "very_short_bodies_under_50_words": int((df_report["body_words"] < 50).sum()),
    "very_long_bodies_over_1000_words": int((df_report["body_words"] > 1000).sum()),
    "rows_with_links_in_body": int((df_report["link_count"] > 0).sum()),
}

display(pd.DataFrame([duplicate_report]).T.rename(columns={0: "count"}))

print("\n📌 Important noise pattern counts by source:")

PATTERN_CHECKS = {
    "bbc_xigashada_sawirka": r"Xigashada Sawirka",
    "bbc_end_of_content": r"End of content",
    "bbc_ugu_akhris_badan": r"Ugu akhris badan",
    "bbc_whatsapp_ad": r"WhatsApp|Halkaan kaga soo biir|Dhamaadka xayeysiinta",

    "caasimada_share_dateline": r"^Share\s+",
    "caasimada_read_more": r"Read more",
    "caasimada_online": r"Caasimada\s+Online",
    "caasimada_isha": r"Isha\s*:",
    "caasimada_wq_credit": r"W/[QT]\s*:",
    "caasimada_email": r"\[email|email\s+protected",

    "goobjoog_news": r"Goobjoog\s+News",
    "goobjoog_halkaan_hoose": r"Halkaan\s+hoose|Hoos\s+Ka\s+Dhageyso",
    "goobjoog_wariye": r"Wariye\s*:",

    "mustaqbal_tail_or_name": r"\bMustaqbal\b",
    "mustaqbal_isha": r"Isha\s*:",
    "mustaqbal_lasoco": r"La\s+soco\s+wixii",

    "kooxda_site": r"Kooxda\.com",
    "kubadlive_views": r"Views?\s*:\s*\d+",
    "laacibnet_comment_form": r"Save my name, email, and website in this browser",
    "puntlandpost_tail": r"PUNTLAND\s+POST",
    "sonna_by_credit": r"\bBy\.?\s+",
    "wararka24_source": r"Wararka24",
    "garowe_boilerplate": r"Advertise with GaroweOnline|manage your cookie choices|Copyright © GAROWONLINE",

    "policy_contact_pages": r"Privacy Policy|Terms and Conditions|Editorial Policy|Cookie Policy|Contact Us|nala soo xiriir|Disclaimer",
    "contact_form": r"Your name\s+Your email\s+Subject",
}

pattern_rows = []

for site, sub in df_report.groupby("site"):
    for pattern_name, pattern in PATTERN_CHECKS.items():
        body_count = sub["body"].astype(str).str.contains(pattern, case=False, regex=True).sum()
        headline_count = sub["headline"].astype(str).str.contains(pattern, case=False, regex=True).sum()

        total = int(body_count + headline_count)

        if total > 0:
            pattern_rows.append({
                "site": site,
                "pattern": pattern_name,
                "count": total,
                "percent_of_site": round(total / len(sub) * 100, 2),
            })

pattern_report = pd.DataFrame(pattern_rows)

if len(pattern_report):
    pattern_report = pattern_report.sort_values(["site", "count"], ascending=[True, False])
    display(pattern_report)
else:
    print("No known noise patterns found by the current pattern checks.")

════════════════════════════════════════════════════════════════════════════════
RAW DATASET OVERVIEW
════════════════════════════════════════════════════════════════════════════════
Total rows: 35,797
Total sources: 12
Total categories: 4

📌 Rows by source:


,site,rows
0,goobjoog,12860
1,caasimada,9365
2,kooxda,7193
3,mustaqbalmedia,4640
4,laacibnet,1094
5,bbc_somali,245
6,puntlandpost,226
7,kubadlive,114
8,sonna,26
9,wararka24,15



📌 Rows by category:


,category,rows
0,caalamka,10116
1,siyaasad,10062
2,ciyaaro,9024
3,amni,6595



📌 Missing / empty values:


...

,site,count,mean,std,min,25%,50%,75%,max
0,bbc_somali,245.0,662.6,302.1,20.0,472.0,645.0,831.0,1932.0
1,caasimada,9365.0,386.5,231.9,13.0,248.0,310.0,436.0,2692.0
2,garoweonline,11.0,232.0,61.7,46.0,250.0,250.0,250.0,256.0
3,goobjoog,12860.0,177.4,153.0,9.0,102.0,138.0,198.0,2392.0
4,kooxda,7193.0,170.8,119.4,12.0,114.0,145.0,190.0,2453.0
5,kooxdamanta,8.0,329.1,85.7,140.0,321.5,340.5,373.2,429.0
6,kubadlive,114.0,287.8,159.1,10.0,171.0,259.5,371.5,941.0
7,laacibnet,1094.0,218.7,187.1,91.0,160.0,185.0,220.0,2577.0
8,mustaqbalmedia,4640.0,218.8,111.9,8.0,162.0,193.0,239.0,1655.0
9,puntlandpost,226.0,207.3,113.6,44.0,139.0,189.0,240.8,1072.0



📌 Duplicate report:


,count
duplicate_url_rows,1407
unique_duplicate_urls,699
duplicate_exact_body_rows,1442
very_short_bodies_under_50_words,411
very_long_bodies_over_1000_words,432
rows_with_links_in_body,119



📌 Important noise pattern counts by source:


,site,pattern,count,percent_of_site
2,bbc_somali,bbc_ugu_akhris_badan,225,91.84
0,bbc_somali,bbc_xigashada_sawirka,222,90.61
3,bbc_somali,bbc_whatsapp_ad,155,63.27
1,bbc_somali,bbc_end_of_content,148,60.41
5,bbc_somali,mustaqbal_tail_or_name,6,2.45
...,...,...,...,...
59,sonna,mustaqbal_tail_or_name,2,7.69
63,wararka24,wararka24_source,10,66.67
64,wararka24,policy_contact_pages,4,26.67
62,wararka24,sonna_by_credit,2,13.33


## Cell 3 — Advanced Cleaning Functions and Site Dispatch System

In [3]:
# CELL 3 — ADVANCED CLEANING FUNCTIONS + SITE DISPATCH SYSTEM
# ===========================================================

def safe_text(x) -> str:
    """Converts missing/non-string values into safe strings."""
    if pd.isna(x):
        return ""
    return str(x)


def normalize_text(text: str) -> str:
    """
    Safe general text normalization.
    It keeps Somali text but removes invisible characters and spacing problems.
    """
    text = safe_text(text)
    text = html.unescape(text)

    # Remove invisible characters
    text = re.sub(r"[\u200b\u200c\u200d\ufeff\u00ad]", "", text)

    # Normalize whitespace
    text = re.sub(r"[\r\n\t\xa0]+", " ", text)
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    text = re.sub(r"([.,;:!?]){2,}", r"\1", text)
    text = re.sub(r"\s{2,}", " ", text)

    return text.strip()


def clean_headline_v2(text: str) -> str:
    """
    Conservative headline cleaning.
    Removes media labels and site-name suffixes only.
    """
    text = normalize_text(text)

    text = re.sub(
        r"^(Sawirro|Sawir|Daawo|DAAWO|Muqaal|VIDEO|PHOTO|Akhriso|AKHRISO|Akhri|Maqal)\s*[:\-–—]?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*[|\-–—]\s*(Caasimada(\s+Online)?|Goobjoog(\s+News)?|Mustaqbal(\s+Media)?|"
        r"Kooxda\.com|Kubadlive|Laacibnet|Puntland\s+Post|SONNA|Wararka24|BBC).*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"^[\"“”'‘’]+|[\"“”'‘’]+$", "", text)

    return normalize_text(text)


def remove_urls_and_emails(text: str) -> str:
    """
    Removes emails, obfuscated emails, and raw URLs from article bodies.
    """
    text = safe_text(text)

    text = re.sub(r"\[email[^\]]*\]|\bemail\s+protected\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text, flags=re.IGNORECASE)

    return text


def remove_contact_form_text(text: str) -> str:
    """
    Removes common WordPress comment/contact form text.
    """
    text = safe_text(text)

    text = re.sub(
        r"Your name\s+Your email\s+Subject\s+Your message(?:\s+optional)?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Save my name,\s*email,\s*and website in this browser for the next time I comment\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_safe_social_cta(text: str) -> str:
    """
    Removes short social media call-to-action sentences.
    This is conservative: it targets CTA phrases, not normal article paragraphs.
    """
    text = safe_text(text)

    text = re.sub(
        r"(Warbixinada qotada dheer iyo wararka BBC Somali[^.]{0,180}?WhatsApp\.?\s*)",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b(Halkaan kaga soo biir|Dhamaadka xayeysiinta|follow us|subscribe|telegram channel|whatsapp channel)\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


# ─────────────────────────────────────────────────────────────────────────────
# Site-specific cleaners
# ─────────────────────────────────────────────────────────────────────────────

def remove_bbc_boilerplate(text: str) -> str:
    """
    BBC Somali cleaner.

    Important:
    We remove 'End of content' as a marker only.
    We DO NOT remove everything after it because useful article text sometimes continues.
    """
    text = safe_text(text)

    bbc_credits = (
        r"Getty Images?|Reuters|AFP|EPA|AP|PA Media|BBC|FIFA|SONNA|CAFONLINE|Cafonline|"
        r"Instagram|Facebook|X|G\.D\.|Villa|Baraha Bulshada|Social Media|FC Barcelona"
    )

    text = re.sub(
        rf"Xigashada Sawirka\s*,?\s*(?:{bbc_credits})?\s*\.?\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bEnd of content\b|\bEnd of Ugu akhris badan\b|\bUgu akhris badan\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Warbixinada qotada dheer iyo wararka BBC Somali[^.]{0,180}?WhatsApp\.?\s*"
        r"(?:Halkaan kaga soo biir)?\s*(?:Dhamaadka xayeysiinta)?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_caasimada_boilerplate(text: str) -> str:
    """
    Caasimada Online cleaner.
    Removes datelines, Read more, source credits, bylines, and video prompts.
    """
    text = safe_text(text).strip()

    # Example: Share Muqdisho (Caasimada Online) –
    text = re.sub(
        r"^Share\s+.{0,120}?(?:Caasimada\s+Onl(?:ine|ien)|Caasimada\s+Online\)?)[^–—-]{0,20}[–—-]\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Fallback for malformed Share datelines
    text = re.sub(
        r"^Share\s+[A-Za-zÀ-ÖØ-öø-ÿ\s().,'-]{1,80}\s*[–—-]\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\bRead more\b", " ", text, flags=re.IGNORECASE)

    text = re.sub(
        r"Isha\s*:\s*[A-Za-z][A-Za-z\s/&+.-]{1,80}",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bW/[QT]\s*:\s*[^.]{0,160}",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Hoos\s+ka\s+daawo[^.]*\.?|Halkan\s+ka\s+daawo[^.]*\.?|HALKAN\s+CLICK\s+SII",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Caasimada\s+Online\s+Xafiiska\s+[\w\s]{1,40}\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Caasimada\s+Online,?\s+waa\s+mareeg[^.]{0,250}(?:Mahadsanid\.?)?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Caasimada\s+Online\s+waxay\s+leedahay\s+App[^.]{0,250}",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Wire-service credits at the end
    text = re.sub(
        r"(?:AP|VOA|AFP|Reuters|BBC)(?:\s*[,+/&]\s*(?:AP|VOA|AFP|Reuters|BBC|Somali|SOMALI))*\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_goobjoog_boilerplate(text: str) -> str:
    """
    Goobjoog cleaner.
    Removes Goobjoog tail credits and audio/video prompts.
    """
    text = safe_text(text)

    text = re.sub(
        r"warkaan\s+wixii\s+kusoo?\s+kordha\s+kala\s+soco\s+wararkeena\s+kale",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"wixii\s+ku\s+soo\s+kordha\s+kala\s+soco\s+Goobjoog\.?com",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"(Halkaan\s+hoose\s+ka\s+(?:daawo|dhageyso)|Hoos\s+Ka\s+Dhageyso|HALKA\s+HOOSE\s+KA\s+DHAGEYSO)[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bWariye\s*:\s*[^.]{0,80}\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bGoobjoog\s+News\b\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_mustaqbal_boilerplate(text: str) -> str:
    """
    Mustaqbal Media cleaner.
    """
    text = safe_text(text)

    text = re.sub(
        r"La\s+soco\s+wixii\s+kasoo\s+kordha\s+Mustaqbal\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Isha\s*:\s*[A-Za-z][A-Za-z\s/&+.-]{1,80}\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"aragtida\s+.*?kamana\s+turjumayaan\s+aragtida\s+Mustaqbal\s+Media\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bMustaqbal\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_kooxda_boilerplate(text: str) -> str:
    """
    Kooxda.com cleaner.
    """
    text = safe_text(text)

    text = re.sub(
        r"Kooxda\.com\s+ayaa\s+xusaysa[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"La\s+soco\s+Kooxda\.com[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Sawirro?\s+la\s+qaaday[^.]*\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bKooxda\.com\b\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_kubadlive_boilerplate(text: str) -> str:
    """
    Kubadlive cleaner.
    """
    text = safe_text(text)

    text = re.sub(r"\bViews?\s*:\s*\d+\b", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"Halkan\s+ka\s+daawo\s*:-?", " ", text, flags=re.IGNORECASE)

    return text


def remove_laacibnet_boilerplate(text: str) -> str:
    """
    Laacibnet cleaner.
    """
    text = safe_text(text)

    text = re.sub(
        r"Save my name,\s*email,\s*and website in this browser for the next time I comment\.?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\bLaacibnet\b\s*$", " ", text, flags=re.IGNORECASE)

    return text


def remove_puntlandpost_boilerplate(text: str) -> str:
    """
    Puntland Post cleaner.
    """
    text = safe_text(text)

    text = re.sub(r"\bPUNTLAND\s+POST\b\s*$", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\bSii\s+akhri\b\s*$", " ", text, flags=re.IGNORECASE)

    return text


def remove_sonna_boilerplate(text: str) -> str:
    """
    SONNA cleaner.
    Removes source dateline at the beginning and author credit at the end.
    """
    text = safe_text(text)

    text = re.sub(
        r"^[A-ZÀ-ÖØ-Ýa-zà-öø-ÿ’'\-\s]{2,35}\s*\(?F?SONNA\)?\s*[-:–—]*\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bBy\.?\s+[A-Za-zÀ-ÖØ-öø-ÿ/.'\-\s]{2,60}\s*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


def remove_wararka24_boilerplate(text: str) -> str:
    """
    Wararka24 cleaner.
    """
    text = safe_text(text)

    text = re.sub(
        r"^Wararka24\s+\d{1,2}[-/]\d{1,2}[-/]\d{4}\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^[A-ZÀ-ÖØ-Ýa-zà-öø-ÿ’'\-\s]{2,35}\s*[-–—]\s*Wararka24\s*[-–—]?\s*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Wararka24\s*[–—-]\s*Xog rasmi ah[^.]*$",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\bWararka24\.com\b", " ", text, flags=re.IGNORECASE)

    return text


def remove_garoweonline_boilerplate(text: str) -> str:
    """
    GaroweOnline cleaner.
    Most GaroweOnline rows in this dataset appear to be listing/cookie pages.
    """
    text = safe_text(text)

    text = re.sub(
        r"Copyright\s+©\s+GAROWONLINE\s+All\s+Rights\s+Reserved",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"Advertise with GaroweOnline[^.]*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"To manage your cookie choices[^.]*",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


# Dispatch system: each website gets only the cleaners it needs
SITE_CLEANERS = {
    "bbc_somali": [remove_bbc_boilerplate],
    "caasimada": [remove_caasimada_boilerplate],
    "goobjoog": [remove_goobjoog_boilerplate],
    "mustaqbalmedia": [remove_mustaqbal_boilerplate],
    "kooxda": [remove_kooxda_boilerplate],
    "kooxdamanta": [],
    "kubadlive": [remove_kubadlive_boilerplate],
    "laacibnet": [remove_laacibnet_boilerplate],
    "puntlandpost": [remove_puntlandpost_boilerplate],
    "sonna": [remove_sonna_boilerplate],
    "wararka24": [remove_wararka24_boilerplate],
    "garoweonline": [remove_garoweonline_boilerplate],
}


def clean_body_v2(text: str, site: str = "") -> str:
    """
    Full body cleaning pipeline.

    Steps:
    1. General normalization
    2. Site-specific cleaners
    3. General boilerplate cleanup
    4. Final normalization
    """
    text = normalize_text(text)
    site = safe_text(site).strip().lower()

    for fn in SITE_CLEANERS.get(site, []):
        text = fn(text)

    text = remove_contact_form_text(text)
    text = remove_urls_and_emails(text)
    text = remove_safe_social_cta(text)

    # Remove trailing ellipsis from teaser-like rows
    text = re.sub(r"\s*[…]+\s*$", "", text)
    text = re.sub(r"\s*\.{3,}\s*$", "", text)

    return normalize_text(text)


print("✅ Advanced cleaning functions loaded.")
print(f"Configured sites: {list(SITE_CLEANERS.keys())}")

✅ Advanced cleaning functions loaded.
Configured sites: ['bbc_somali', 'caasimada', 'goobjoog', 'mustaqbalmedia', 'kooxda', 'kooxdamanta', 'kubadlive', 'laacibnet', 'puntlandpost', 'sonna', 'wararka24', 'garoweonline']


## Cell 4 — Apply Cleaning, Validate Rows, and Deduplicate

In [4]:
# CELL 4 — APPLY ADVANCED CLEANING + VALIDATION FLAGS
# ===================================================

work_df = raw_df.copy()

for col in ["site", "category", "url", "headline", "body"]:
    if col not in work_df.columns:
        work_df[col] = ""

work_df["site"] = work_df["site"].astype(str).str.strip().str.lower()
work_df["category"] = work_df["category"].astype(str).str.strip().str.lower()

# Keep originals for inspection
work_df["headline_original"] = work_df["headline"].astype(str)
work_df["body_original"] = work_df["body"].astype(str)

# Clean headline and body
print("🧹 Cleaning headlines...")
work_df["headline_clean"] = work_df["headline_original"].apply(clean_headline_v2)

print("🧹 Cleaning bodies...")
work_df["body_clean"] = work_df.apply(
    lambda row: clean_body_v2(row["body_original"], row["site"]),
    axis=1
)

# Length columns
work_df["headline_chars_before"] = work_df["headline_original"].str.len()
work_df["headline_chars_after"] = work_df["headline_clean"].str.len()

work_df["body_chars_before"] = work_df["body_original"].str.len()
work_df["body_chars_after"] = work_df["body_clean"].str.len()

work_df["body_words_before"] = work_df["body_original"].str.split().str.len()
work_df["body_words_after"] = work_df["body_clean"].str.split().str.len()

work_df["cleaning_loss_ratio"] = 1 - (
    work_df["body_chars_after"] / work_df["body_chars_before"].replace(0, np.nan)
)

work_df["cleaning_loss_ratio"] = work_df["cleaning_loss_ratio"].fillna(0)

# ─────────────────────────────────────────────────────────────────────────────
# Article-level removal rules
# These mark rows that are very likely not real article content.
# ─────────────────────────────────────────────────────────────────────────────

POLICY_HEADLINE_RE = re.compile(
    r"^(privacy\s*policy|terms\s+(and\s+conditions|of\s+service)|editorial\s+policy|"
    r"cookie\s+policy|contact\s+us|nala\s+soo\s+xiriir|adverti[sz]e\s+with\s+us|"
    r"bookmark\s+page|disclaimer|guidance:\s*links\s+and\s+feeds)$",
    re.IGNORECASE
)

POLICY_URL_RE = re.compile(
    r"/(privacy|terms|cookie|contact|contactus|nala-soo-xiriir|advertise|advertize|disclaimer|editorial|bookmark)",
    re.IGNORECASE
)

GOOBJOOG_NAV_TITLES = {
    "Wararka Caalamka", "Dhaqanka & Buugta", "Aragtida Xorta Ah",
    "Arrimaha Ganacsiga", "La-dagaallanka Al-Shabaab", "Deegaanka & Cimilada",
    "Arrimaha Amniga", "Dagaalka Dib-u-Xoreynta", "Siyaasadda Xogdhawrka",
    "Wararka Maanta", "Ciyaaraha", "Caafimaad", "Diinta",
}


def get_article_removal_reason(row) -> str:
    """
    Returns a reason if the whole row should be removed.
    Otherwise returns empty string.
    """
    site = safe_text(row.get("site", "")).strip().lower()
    headline = safe_text(row.get("headline_original", "")).strip()
    url = safe_text(row.get("url", "")).strip()
    body = safe_text(row.get("body_original", "")).strip()

    if headline in GOOBJOOG_NAV_TITLES:
        return "navigation_page"

    if POLICY_HEADLINE_RE.search(headline):
        return "policy_or_contact_page"

    if POLICY_URL_RE.search(url):
        return "policy_or_contact_url"

    if re.search(r"Your name\s+Your email\s+Subject", body, re.IGNORECASE):
        return "contact_form_page"

    if site == "garoweonline" and re.search(
        r"Advertise with GaroweOnline|manage your cookie choices|Copyright © GAROWONLINE",
        body,
        re.IGNORECASE
    ):
        return "garoweonline_listing_or_cookie_page"

    if re.search(
        r"Xuquuqda Gaarka ah ee CCPA|Ha Iibiina Macluumaadkayga",
        body,
        re.IGNORECASE
    ):
        return "privacy_ccpa_page"

    return ""


print("🔎 Marking non-article rows...")
work_df["removal_reason"] = work_df.apply(get_article_removal_reason, axis=1)


def normalized_hash(text: str) -> str:
    text = normalize_text(text).lower()
    return hashlib.md5(text.encode("utf-8")).hexdigest()


work_df["url_norm"] = work_df["url"].astype(str).str.strip().str.lower()
work_df["body_hash"] = work_df["body_clean"].apply(normalized_hash)

work_df["duplicate_url"] = (
    work_df["url_norm"].ne("") &
    work_df.duplicated("url_norm", keep=False)
)

work_df["duplicate_clean_body"] = (
    work_df["body_clean"].str.len().gt(50) &
    work_df.duplicated("body_hash", keep=False)
)


# ─────────────────────────────────────────────────────────────────────────────
# Suspicious flags
# ─────────────────────────────────────────────────────────────────────────────

SOMALI_MARKERS = {
    "ayaa", "ayuu", "ayey", "waxaa", "waxuu", "waxay", "waxan",
    "oo", "iyo", "ee", "ku", "ka", "la", "uu", "ay",
    "in", "waa", "ah", "ama", "haddii", "sidoo", "kale",
    "waxa", "lagu", "looga", "ugu", "kaga", "iska",
    "dalka", "magaalada", "dowladda", "shacabka", "caalamka",
    "madaxweynaha", "wasaaradda", "ciidanka", "kooxda",
}

ENGLISH_MARKERS = {
    "the", "and", "is", "in", "of", "to", "a", "that",
    "it", "for", "with", "on", "are", "was", "has",
    "have", "this", "an", "at"
}


def marker_ratio(text: str, markers: set) -> float:
    words = re.findall(r"\b\w+\b", safe_text(text).lower())

    if not words:
        return 0

    return sum(1 for w in words if w in markers) / len(words)


def headline_inside_body(row) -> bool:
    headline = normalize_text(row.get("headline_clean", "")).lower()
    body = normalize_text(row.get("body_clean", "")).lower()

    if len(headline) < 20:
        return False

    return headline in body[:600]


def make_suspicious_flags(row) -> list:
    flags = []

    body_clean = safe_text(row.get("body_clean", ""))
    headline_clean = safe_text(row.get("headline_clean", ""))
    body_words = int(row.get("body_words_after", 0))
    link_count = len(re.findall(r"https?://|www\.", safe_text(row.get("body_original", ""))))

    if row.get("removal_reason", ""):
        flags.append(f"remove:{row.get('removal_reason')}")

    if not headline_clean.strip():
        flags.append("empty_headline_after_cleaning")

    if not body_clean.strip():
        flags.append("empty_body_after_cleaning")

    if len(headline_clean) < 8:
        flags.append("very_short_headline")

    if body_words < 50:
        flags.append(f"very_short_body({body_words}_words)")

    if body_words > 1000:
        flags.append(f"very_long_body({body_words}_words)")

    if row.get("cleaning_loss_ratio", 0) > 0.65:
        flags.append(f"cleaning_removed_too_much({row.get('cleaning_loss_ratio'):.0%})")

    if row.get("duplicate_url", False):
        flags.append("duplicate_url")

    if row.get("duplicate_clean_body", False):
        flags.append("duplicate_clean_body")

    if link_count > 2:
        flags.append(f"too_many_links({link_count})")

    if headline_inside_body(row):
        flags.append("headline_repeated_inside_body")

    so_ratio = marker_ratio(body_clean, SOMALI_MARKERS)
    en_ratio = marker_ratio(body_clean, ENGLISH_MARKERS)

    if body_words >= 50 and so_ratio < 0.015 and en_ratio > 0.10:
        flags.append(f"possibly_non_somali(so={so_ratio:.2f},en={en_ratio:.2f})")

    residual_boilerplate = re.search(
        r"Read more|Xigashada Sawirka|Goobjoog News|PUNTLAND POST|Save my name|Views?\s*:\s*\d+|"
        r"Advertise with GaroweOnline|manage your cookie choices",
        body_clean,
        re.IGNORECASE
    )

    if residual_boilerplate:
        flags.append("residual_boilerplate_after_cleaning")

    return flags


print("🚩 Creating suspicious flags...")
work_df["suspicious_flags"] = work_df.apply(make_suspicious_flags, axis=1)
work_df["suspicious_reason"] = work_df["suspicious_flags"].apply(lambda flags: "; ".join(flags))
work_df["is_suspicious"] = work_df["suspicious_flags"].apply(lambda flags: len(flags) > 0)


# ─────────────────────────────────────────────────────────────────────────────
# Build final cleaned dataset
# ─────────────────────────────────────────────────────────────────────────────

cleaned_dataset = work_df.copy()

# Remove rows that are clearly not articles
cleaned_dataset = cleaned_dataset[cleaned_dataset["removal_reason"].eq("")].copy()

# Remove empty cleaned values
cleaned_dataset = cleaned_dataset[
    cleaned_dataset["headline_clean"].str.strip().ne("") &
    cleaned_dataset["body_clean"].str.strip().ne("")
].copy()

# Remove very short bodies from final dataset.
# They remain available in suspicious_rows_for_review.
cleaned_dataset = cleaned_dataset[cleaned_dataset["body_words_after"] >= 50].copy()

# Deduplicate by URL, but do not collapse empty URLs together
with_url = cleaned_dataset[cleaned_dataset["url_norm"].ne("")].drop_duplicates("url_norm", keep="first")
without_url = cleaned_dataset[cleaned_dataset["url_norm"].eq("")]
cleaned_dataset = pd.concat([with_url, without_url], ignore_index=True)

# Deduplicate by cleaned body hash
cleaned_dataset = cleaned_dataset.drop_duplicates("body_hash", keep="first").copy()

# Final training-friendly columns
cleaned_dataset["headline"] = cleaned_dataset["headline_clean"]
cleaned_dataset["body"] = cleaned_dataset["body_clean"]

print("\n" + "═" * 80)
print("✅ ADVANCED CLEANING COMPLETE")
print("═" * 80)

print(f"Raw rows: {len(raw_df):,}")
print(f"Rows after cleaning/removal/deduplication: {len(cleaned_dataset):,}")
print(f"Suspicious rows for review: {work_df['is_suspicious'].sum():,}")

print("\n📌 Removal reasons:")
display(
    work_df["removal_reason"]
    .replace("", "kept")
    .value_counts()
    .rename_axis("reason")
    .reset_index(name="rows")
)

print("\n📌 Cleaned rows by source:")
display(
    cleaned_dataset["site"]
    .value_counts()
    .rename_axis("site")
    .reset_index(name="rows")
)

print("\n📌 Cleaned rows by category:")
display(
    cleaned_dataset["category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="rows")
)

print("\n📌 Preview cleaned dataset:")
display(cleaned_dataset[["site", "category", "headline", "body"]].head())

🧹 Cleaning headlines...
🧹 Cleaning bodies...
🔎 Marking non-article rows...
🚩 Creating suspicious flags...

════════════════════════════════════════════════════════════════════════════════
✅ ADVANCED CLEANING COMPLETE
════════════════════════════════════════════════════════════════════════════════
Raw rows: 35,797
Rows after cleaning/removal/deduplication: 34,550
Suspicious rows for review: 3,731

📌 Removal reasons:


,reason,rows
0,kept,35745
1,navigation_page,19
2,policy_or_contact_page,18
3,garoweonline_listing_or_cookie_page,10
4,policy_or_contact_url,5



📌 Cleaned rows by source:


,site,rows
0,goobjoog,12346
1,caasimada,8755
2,kooxda,7183
3,mustaqbalmedia,4563
4,laacibnet,1091
5,bbc_somali,236
6,puntlandpost,222
7,kubadlive,109
8,sonna,24
9,wararka24,13



📌 Cleaned rows by category:


,category,rows
0,caalamka,10010
1,siyaasad,9308
2,ciyaaro,8961
3,amni,6271



📌 Preview cleaned dataset:


,site,category,headline,body
0,bbc_somali,ciyaaro,Maxay dalalku ka faa'idaan marka ay u soo baxaan ka qeybgalka Koobka Adduunka?,"Dalalka si guul leh ugu soo baxa Koobka Adduunka ee FIFA waxay helaan sharaf weyn oo ay dunidu u hayso, sidoo kale faa'iidooyin dhaqaale iyo farxad weyn ayaa soo wajahda shacabkooda. Tani ma aha oo keliya guul ciyaareed, balse waa guul saameyn ku..."
1,bbc_somali,ciyaaro,Maxaa ka jira in ciyaartooyda kooxdii caanka ahayd ee Real Madrid mid mid isu dilayaan?,"Kooxda Real Madrid ayaa toddobaadkan gashay xaalad qalalaase weyn ah, xilli ay ahayd in dhammaan diiradda lagu saaro kulanka adag ee El Clasico ee ay la ciyaarayaan kooxda ay sida weyn u xafiiltamaan ee FC Barcelona. Halkii laga filayay in kooxda..."
2,bbc_somali,ciyaaro,Ciyaaryahanada ugu sarreeya Afrika ee aan ka qayb galayn Koobka Adduunka 2026,"Iyada oo ay soo dhowaatay isku-aadka Koobka Adduunka, dalal badan, gaar ahaan kuwa ka qayb galaya koobka ayaa bilaabay qorshahooda inay dhowaan safraan. Sannadka, 48 dal ayaa ka qayb galaya tartanka oo ay si wadajir ah u martigalinayaan Maraykank..."
3,bbc_somali,ciyaaro,Maxaa ugu wacan Arsenal inaysan weligeed ku guulaysan Champions League,"Arsenal waligeed kuma guuleysan Champions League iyo Europa League. Hadda waxay soo gaartay Semifinalka, waxayna barbarro 1-1 ah la gashay Atletico Madrid oo ay marti ugu ahayd gurigeeda lugtii hore ee labada cayaarood ee semifinalka, ayadoo lugt..."
4,bbc_somali,ciyaaro,Dhamaan xogta iyo macluumaadka aad u baahan tahay in aad ka ogaato koobka kubadda cagta adduunka ee 2026,"Iyadoo dhamaan kooxaha ka qayb galaya koobka adduunka hadda la og yahay, indhaha ayaa u soo jeestay tartankii ugu weynaa ee Koobka Adduunka ee abid la qabto. Lixdii kooxood ee ugu dambeeyay ayaa dhawaan u soo baxay Koobka Adduunka 2026, sidaas da..."


## Cell 5 — Before/After Cleaning Inspection

In [5]:
# CELL 5 — BEFORE / AFTER INSPECTION
# ==================================

def get_removed_chunks(original: str, cleaned: str, max_chunks: int = 6) -> str:
    """
    Shows text chunks that disappeared after cleaning.
    This is only for inspection, not for final processing.
    """
    original = normalize_text(original)
    cleaned = normalize_text(cleaned)

    matcher = SequenceMatcher(None, original, cleaned)
    removed = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag in ["delete", "replace"]:
            chunk = original[i1:i2].strip()

            if len(chunk) >= 20:
                removed.append(chunk)

        if len(removed) >= max_chunks:
            break

    return " | ".join(removed)


def inspect_cleaning(site=None, category=None, suspicious_only=False, n=10, random_state=42):
    """
    Shows before/after cleaning examples.

    Usage examples:
      inspect_cleaning(site="bbc_somali", n=5)
      inspect_cleaning(site="caasimada", suspicious_only=True, n=10)
      inspect_cleaning(category="ciyaaro", n=10)
    """
    temp = work_df.copy()

    if site is not None:
        temp = temp[temp["site"].eq(site)]

    if category is not None:
        temp = temp[temp["category"].eq(category)]

    if suspicious_only:
        temp = temp[temp["is_suspicious"]]

    if temp.empty:
        print("No rows found for this filter.")
        return

    sample = temp.sample(min(n, len(temp)), random_state=random_state).copy()

    sample["removed_text_sample"] = sample.apply(
        lambda row: get_removed_chunks(row["body_original"], row["body_clean"]),
        axis=1
    )

    cols = [
        "site",
        "category",
        "headline_original",
        "headline_clean",
        "body_words_before",
        "body_words_after",
        "cleaning_loss_ratio",
        "suspicious_reason",
        "removed_text_sample",
        "body_original",
        "body_clean",
    ]

    display(sample[cols])


# Example inspections
inspect_cleaning(site="bbc_somali", n=3)
inspect_cleaning(site="caasimada", n=3)
inspect_cleaning(site="laacibnet", n=3)

,site,category,headline_original,headline_clean,body_words_before,body_words_after,cleaning_loss_ratio,suspicious_reason,removed_text_sample,body_original,body_clean
24,bbc_somali,ciyaaro,"BBC News , World Service - Home","BBC News, World Service - Home",114,114,0.000000,,,"BBC News is the world’s most trusted international news organisation with a weekly audience of 418 million people. BBC World Service - in 43 languages with journalists in 64 countries - reaches 313 million people every week with its independent, ...","BBC News is the world’s most trusted international news organisation with a weekly audience of 418 million people. BBC World Service - in 43 languages with journalists in 64 countries - reaches 313 million people every week with its independent, ..."
6,bbc_somali,ciyaaro,Maxay kala kulmeen Australia ciyaartoydii reer Iran ee iska dhiibay?,Maxay kala kulmeen Australia ciyaartoydii reer Iran ee iska dhiibay?,419,391,0.063986,,End of Ugu akhris badan End of content Warbixinada qotada dheer iyo wararka BBC Somali oo toos kuugu imanaaya WhatsApp. Halkaan kaga soo biir Dhamaadka xayeysiinta,"Xigashada Sawirka, jack tan Laba ciyaartoy oo kubadda cagta ah oo ka soo jeeda Iiraan, kuwaas oo kal hore magangelyo ka helay Australia ayaa sheegay in dalkaasi uu siiyey ""rejo mustaqbalka ay si nabad ah ugu noolaan karaan uguna ciyaari karaan."" ...","jack tan Laba ciyaartoy oo kubadda cagta ah oo ka soo jeeda Iiraan, kuwaas oo kal hore magangelyo ka helay Australia ayaa sheegay in dalkaasi uu siiyey ""rejo mustaqbalka ay si nabad ah ugu noolaan karaan uguna ciyaari karaan."" Atefeh Ramezanisade..."
153,bbc_somali,ciyaaro,Weeraryahanka Kenya ku dhashay ee Soomaaliya uga qeybgalaya cayaaraha caalamiga ah,Weeraryahanka Kenya ku dhashay ee Soomaaliya uga qeybgalaya cayaaraha caalamiga ah,471,441,0.065098,,"End of Ugu akhris badan | End of content Xigashada Sawirka, | Warbixinada qotada dheer iyo wararka BBC Somali oo toos kuugu imanaaya WhatsApp. Halkaan kaga soo biir Dhamaadka xayeysiinta","Xigashada Sawirka, Syracuse Orange Weerar yahanka Kenya ku dhashay ee Cabdi Muya Salim oo u ciyaara kooxda Orlando City ee Mareykanka ka dhisan ayaa looga wacay xulka qaranka Soomaaliya si uu uga qeybgalo is reebreebka cayaaraha Koobka Adduunka. ...",Syracuse Orange Weerar yahanka Kenya ku dhashay ee Cabdi Muya Salim oo u ciyaara kooxda Orlando City ee Mareykanka ka dhisan ayaa looga wacay xulka qaranka Soomaaliya si uu uga qeybgalo is reebreebka cayaaraha Koobka Adduunka. Soomaaliya ayaa ku ...


,site,category,headline_original,headline_clean,body_words_before,body_words_after,cleaning_loss_ratio,suspicious_reason,removed_text_sample,body_original,body_clean
2036,caasimada,caalamka,Doorasho maanta ka socota dalka Jabuuti xilli la hadal-hayo cidda beddeleysa Geelle,Doorasho maanta ka socota dalka Jabuuti xilli la hadal-hayo cidda beddeleysa Geelle,488,479,0.017892,,Share Jabuuti (Caasimada Online) –,Share Jabuuti (Caasimada Online) – Cod-bixiyayaasha Jabuuti ayaa maanta dooranaya baarlaman cusub ayada oo ay dhaceyso doorasho ay qaadaceen xisbiyada mucaaradka ee ugu weyn. Xisbiga madaxweyne Ismaaciil Cumar Geelle ee talada haya ee Union for P...,"Cod-bixiyayaasha Jabuuti ayaa maanta dooranaya baarlaman cusub ayada oo ay dhaceyso doorasho ay qaadaceen xisbiyada mucaaradka ee ugu weyn. Xisbiga madaxweyne Ismaaciil Cumar Geelle ee talada haya ee Union for Presidential Majority (UMP), ayaa ah..."
4129,caasimada,caalamka,Xiisadda Eritrea iyo Jabuuti oo cirka isku shareertay iyo arrin cusub oo soo baxda,Xiisadda Eritrea iyo Jabuuti oo cirka isku shareertay iyo arrin cusub oo soo baxda,166,157,0.056401,,Share Jabuuti (Caasimada Online) – | [email protected] Read more,"Share Jabuuti (Caasimada Online) – Midowga Afrika ayaa ku baxay xiisada ka dhex aloosan Dowladaha Eritrea iyo Jabuuti kadib markii xadka labadaasi dal laga saaray ciidamada Qadar. Midowga Afrika, ayaa Saraakiil u diray xadka labada dal kuwaa oo q...","Midowga Afrika ayaa ku baxay xiisada ka dhex aloosan Dowladaha Eritrea iyo Jabuuti kadib markii xadka labadaasi dal laga saaray ciidamada Qadar. Midowga Afrika, ayaa Saraakiil u diray xadka labada dal kuwaa oo qiimeynaaya dhaqdhaqaaqa ay wadaan c..."
8922,caasimada,siyaasad,"Daawo: Moole, Joojole, Jeex iyo Banyaa oo la horgeeyay Maxkamadda Ciidamada","Moole, Joojole, Jeex iyo Banyaa oo la horgeeyay Maxkamadda Ciidamada",388,381,0.018563,,Share Muqdisho (Caasimada Online) –,"Share Muqdisho (Caasimada Online) – Maxkamadda Ciidamada Qalaba Sida ayaa waxaa maanta la horgeeyay kiiska afar arday oo dhowaan jirdil xooggan u gaystay macallinkooda oo ku qabtay qish, intii lagu guda jiray imtixaanka shahaadiga dowladda ee san...","Maxkamadda Ciidamada Qalaba Sida ayaa waxaa maanta la horgeeyay kiiska afar arday oo dhowaan jirdil xooggan u gaystay macallinkooda oo ku qabtay qish, intii lagu guda jiray imtixaanka shahaadiga dowladda ee sanadkaan. Maxkamadda Ciidamada ayaa wa..."


,site,category,headline_original,headline_clean,body_words_before,body_words_after,cleaning_loss_ratio,suspicious_reason,removed_text_sample,body_original,body_clean
30278,laacibnet,ciyaaro,Barcelona Oo Rashford U Dirtay Farriin Culus Oo Mustaqbalkiisa Ku Aadan,Barcelona Oo Rashford U Dirtay Farriin Culus Oo Mustaqbalkiisa Ku Aadan,270,255,0.043600,,"Save my name, email, and website in this browser for the next time I comment.","Sida uu qoray suxufiga Guillem Balague ee ka tirsan BBC-da, kooxda Barcelona ayaa fariin u dirtay Marcus Rashford iyadoo ka codsatay inuu dulqaad muujiyo inta ay socdaan wadahadalada qandaraaskiisa. Kooxda ayaa u xaqiijisay Rashford in dib-u-dhac...","Sida uu qoray suxufiga Guillem Balague ee ka tirsan BBC-da, kooxda Barcelona ayaa fariin u dirtay Marcus Rashford iyadoo ka codsatay inuu dulqaad muujiyo inta ay socdaan wadahadalada qandaraaskiisa. Kooxda ayaa u xaqiijisay Rashford in dib-u-dhac..."
29935,laacibnet,ciyaaro,PSG Mise Arsenal? Wesley Sneijder Oo Saadaaliyay Kooxda Ku Guuleysaneysa Champions League-ga,PSG Mise Arsenal? Wesley Sneijder Oo Saadaaliyay Kooxda Ku Guuleysaneysa Champions League-ga,160,145,0.078550,,"Save my name, email, and website in this browser for the next time I comment.","Halyeyga Inter Milan, Wesley Sneijder, ayaa sheegay inuu aaminsan yahay in Arsenal ay 4-0 ku garaaci karto Paris Saint-Germain finalka UEFA Champions League ee ka dhici doona magaalada Budapest dhammaadka bishan. Arsenal ayaa finalka usoo gudubta...","Halyeyga Inter Milan, Wesley Sneijder, ayaa sheegay inuu aaminsan yahay in Arsenal ay 4-0 ku garaaci karto Paris Saint-Germain finalka UEFA Champions League ee ka dhici doona magaalada Budapest dhammaadka bishan. Arsenal ayaa finalka usoo gudubta..."
29884,laacibnet,ciyaaro,Xabi Alonso Oo Chelsea Ka Dalbaday Hal Arrin Ka Hor Intaan La Magacaabin,Xabi Alonso Oo Chelsea Ka Dalbaday Hal Arrin Ka Hor Intaan La Magacaabin,193,178,0.064303,,"Save my name, email, and website in this browser for the next time I comment.","Tababarihii hore ee Bayer Leverkusen iyo Real Madrid, Xabi Alonso, ayaa la sheegay inuu dalab weyn u gudbiyay Chelsea kahor suurtagalnimada uu kula wareegayo kooxda Stamford Bridge. Sida lagu sheegay warbixin kasoo baxday The Touchline, Alonso wu...","Tababarihii hore ee Bayer Leverkusen iyo Real Madrid, Xabi Alonso, ayaa la sheegay inuu dalab weyn u gudbiyay Chelsea kahor suurtagalnimada uu kula wareegayo kooxda Stamford Bridge. Sida lagu sheegay warbixin kasoo baxday The Touchline, Alonso wu..."


## Cell 6 — Inspect Suspicious Rows

In [6]:
# CELL 6 — SUSPICIOUS ROW REVIEW HELPERS
# ======================================

def show_suspicious(reason_contains=None, site=None, n=20, random_state=7):
    """
    Review suspicious rows directly in the notebook.

    Examples:
      show_suspicious(reason_contains="very_short_body", n=20)
      show_suspicious(reason_contains="duplicate", site="caasimada", n=20)
      show_suspicious(site="bbc_somali", n=20)
    """
    temp = work_df[work_df["is_suspicious"]].copy()

    if reason_contains:
        temp = temp[temp["suspicious_reason"].str.contains(reason_contains, case=False, na=False)]

    if site:
        temp = temp[temp["site"].eq(site)]

    if temp.empty:
        print("No suspicious rows found for this filter.")
        return

    sample = temp.sample(min(n, len(temp)), random_state=random_state)

    cols = [
        "site",
        "category",
        "url",
        "headline_original",
        "headline_clean",
        "body_words_before",
        "body_words_after",
        "cleaning_loss_ratio",
        "removal_reason",
        "suspicious_reason",
        "body_original",
        "body_clean",
    ]

    cols = [c for c in cols if c in sample.columns]
    display(sample[cols])


print("✅ Suspicious review helper loaded.")
print("Try: show_suspicious(reason_contains='very_short_body', n=20)")

✅ Suspicious review helper loaded.
Try: show_suspicious(reason_contains='very_short_body', n=20)


## Cell 7 — Final Export

In [8]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]



In [10]:
# CELL 7 — EXPORT CLEANED DATASET + SUSPICIOUS REVIEW FILES
# =========================================================

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Final cleaned dataset
cleaned_export_cols = [
    "id",
    "site",
    "category",
    "url",
    "headline",
    "body",
    "headline_clean",
    "body_clean",
    "body_words_after",
    "source_file",
]

cleaned_export_cols = [c for c in cleaned_export_cols if c in cleaned_dataset.columns]

cleaned_export = cleaned_dataset[cleaned_export_cols].copy()

# Suspicious rows for manual review
suspicious_rows_for_review = work_df[work_df["is_suspicious"]].copy()

review_cols = [
    "id",
    "site",
    "category",
    "url",
    "headline_original",
    "headline_clean",
    "body_words_before",
    "body_words_after",
    "cleaning_loss_ratio",
    "removal_reason",
    "suspicious_reason",
    "body_original",
    "body_clean",
    "source_file",
]

review_cols = [c for c in review_cols if c in suspicious_rows_for_review.columns]
suspicious_export = suspicious_rows_for_review[review_cols].copy()

# Save CSV
cleaned_csv_path = OUTPUT_DIR / "cleaned_dataset.csv"
suspicious_csv_path = OUTPUT_DIR / "suspicious_rows_for_review.csv"

cleaned_export.to_csv(cleaned_csv_path, index=False, encoding="utf-8-sig")
suspicious_export.to_csv(suspicious_csv_path, index=False, encoding="utf-8-sig")

# Save Excel versions for easier manual checking
cleaned_excel_path = OUTPUT_DIR / "cleaned_dataset.xlsx"
suspicious_excel_path = OUTPUT_DIR / "suspicious_rows_for_review.xlsx"

cleaned_export.to_excel(cleaned_excel_path, index=False)
suspicious_export.to_excel(suspicious_excel_path, index=False)

print("✅ Export completed successfully.")
print(f"📁 Cleaned CSV: {cleaned_csv_path.resolve()}")
print(f"📁 Cleaned Excel: {cleaned_excel_path.resolve()}")
print(f"📁 Suspicious CSV: {suspicious_csv_path.resolve()}")
print(f"📁 Suspicious Excel: {suspicious_excel_path.resolve()}")

print("\n📊 Final export summary:")
print(f"Cleaned rows: {len(cleaned_export):,}")
print(f"Suspicious review rows: {len(suspicious_export):,}")

✅ Export completed successfully.
📁 Cleaned CSV: E:\somali_cleaner\data\processed\cleaned_dataset.csv
📁 Cleaned Excel: E:\somali_cleaner\data\processed\cleaned_dataset.xlsx
📁 Suspicious CSV: E:\somali_cleaner\data\processed\suspicious_rows_for_review.csv
📁 Suspicious Excel: E:\somali_cleaner\data\processed\suspicious_rows_for_review.xlsx

📊 Final export summary:
Cleaned rows: 34,550
Suspicious review rows: 3,731


## Notes

Recommended workflow:

1. Run Cell 1 to load your combined raw data.
2. Run Cell 2 to understand the dataset quality.
3. Run Cell 3 to load all cleaning functions.
4. Run Cell 4 to clean, validate, and deduplicate.
5. Run Cell 5 to inspect before/after examples.
6. Run Cell 6 to manually inspect suspicious rows.
7. Run Cell 7 to export the final files.

The cleaning rules are intentionally conservative so they do not remove useful Somali article content accidentally.